# Spectral Fitting with ML Interpolation

This notebook demonstrates fitting transient absorption spectra with Gaussian components and using machine learning to interpolate parameters across time.

## Overview

1. Fit sum of Gaussians to each spectrum
2. Extract amplitude, center, width for each component
3. Train ML model to smooth/interpolate parameters
4. Reconstruct denoised spectra

In [ ]:
# Configuration
from pathlib import Path

# Model parameters
MODEL_TYPE = "knn"  # "knn" or "rf" (random forest)
N_NEIGHBORS = 5     # for KNN

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from dssc import TAData, load_ta_data
from dssc.fitting import (
    fit_ta_gaussians,
    train_parameter_model,
    predict_parameters,
    reconstruct_spectrum,
    sum_of_gaussians,
    GaussianParams,
)
from dssc.constants import wavelength_to_wavenumber
from dssc.plotting import plot_ta_contour

## Step 1: Generate Synthetic TA Data

Create data with known Gaussian components for validation.

In [ ]:
# Create synthetic TA data with 3 Gaussian components
wavelength = np.linspace(420, 750, 150)
time = np.linspace(0.1, 10, 80)
wavenumber = wavelength_to_wavenumber(wavelength)

# True parameters (will vary with time)
# Component 1: GSB at 461 nm (decays with 2 ps)
# Component 2: ESA at 600 nm (decays with 3 ps)
# Component 3: ESA at 680 nm (grows then decays)

signal = np.zeros((len(wavelength), len(time)))

for i, t in enumerate(time):
    # Time-dependent amplitudes
    A1 = -80 * np.exp(-t / 2.0)  # GSB decay
    A2 = 40 * np.exp(-t / 3.0)   # ESA decay
    A3 = 20 * (1 - np.exp(-t / 0.5)) * np.exp(-t / 5.0)  # Rise then decay
    
    # Fixed spectral positions
    params = (
        A1, 1e7/461, 400,   # GSB
        A2, 1e7/600, 600,   # ESA 1
        A3, 1e7/680, 800,   # ESA 2
        0,                   # offset
    )
    
    signal[:, i] = sum_of_gaussians(wavenumber, *params)

# Add noise
np.random.seed(42)
signal += np.random.randn(*signal.shape) * 2

ta_data = TAData(wavelength=wavelength, time=time, signal=signal)

print(f"Data shape: {ta_data.shape}")
print(f"Wavelength: {wavelength.min():.0f} - {wavelength.max():.0f} nm")
print(f"Time: {time.min():.1f} - {time.max():.1f} ps")

In [ ]:
# Visualize the data
fig, ax = plt.subplots(figsize=(10, 6))
plot_ta_contour(ta_data, ax=ax)
ax.set_title('Synthetic TA Data (with noise)')
plt.show()

## Step 2: Fit Gaussians to Each Spectrum

In [ ]:
# Define fitting bounds appropriate for this data
bounds = (
    (-200, 1e7/480, 100, -50, 1e7/650, 100, -50, 1e7/720, 100, -50),
    (50, 1e7/440, 1000, 100, 1e7/550, 1500, 100, 1e7/650, 2000, 50),
)

initial_guess = (
    -50, 1e7/461, 400,
    30, 1e7/600, 600,
    10, 1e7/680, 800,
    0,
)

# Fit all time points
fit_result = fit_ta_gaussians(
    ta_data,
    initial_guess=initial_guess,
    bounds=bounds,
)

print(f"Fit result shape: {fit_result.parameters.shape}")
print(f"Parameters: [A1, μ1, σ1, A2, μ2, σ2, A3, μ3, σ3, C]")

## Step 3: Visualize Fit Parameters vs Time

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Amplitudes
ax = axes[0, 0]
ax.plot(time, fit_result.parameters[0, :], 'b-', label='GSB (461 nm)', linewidth=2)
ax.plot(time, fit_result.parameters[3, :], 'r-', label='ESA1 (600 nm)', linewidth=2)
ax.plot(time, fit_result.parameters[6, :], 'g-', label='ESA2 (680 nm)', linewidth=2)
ax.axhline(0, color='k', linestyle='--', linewidth=0.5)
ax.set_xlabel('Time (ps)')
ax.set_ylabel('Amplitude (mOD)')
ax.set_title('Amplitudes')
ax.legend()

# Centers (convert to wavelength)
ax = axes[0, 1]
ax.plot(time, 1e7/fit_result.parameters[1, :], 'b-', label='GSB', linewidth=2)
ax.plot(time, 1e7/fit_result.parameters[4, :], 'r-', label='ESA1', linewidth=2)
ax.plot(time, 1e7/fit_result.parameters[7, :], 'g-', label='ESA2', linewidth=2)
ax.set_xlabel('Time (ps)')
ax.set_ylabel('Center (nm)')
ax.set_title('Peak Centers')
ax.legend()

# Widths
ax = axes[1, 0]
ax.plot(time, fit_result.parameters[2, :], 'b-', label='GSB', linewidth=2)
ax.plot(time, fit_result.parameters[5, :], 'r-', label='ESA1', linewidth=2)
ax.plot(time, fit_result.parameters[8, :], 'g-', label='ESA2', linewidth=2)
ax.set_xlabel('Time (ps)')
ax.set_ylabel('Width (cm⁻¹)')
ax.set_title('Peak Widths')
ax.legend()

# Example spectrum fit
ax = axes[1, 1]
itime = 20  # Pick a time point
ax.plot(wavelength, ta_data.signal[:, itime], 'ko', markersize=3, label='Data')
reconstructed = reconstruct_spectrum(wavelength, fit_result.parameters[:, itime])
ax.plot(wavelength, reconstructed, 'r-', linewidth=2, label='Fit')
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('ΔA (mOD)')
ax.set_title(f'Spectrum at t = {time[itime]:.1f} ps')
ax.legend()

plt.tight_layout()
plt.show()

## Step 4: Train ML Model for Parameter Interpolation

Use machine learning to smooth noisy parameters and interpolate.

In [ ]:
# Train model
model = train_parameter_model(
    fit_result,
    ta_data,
    model_type=MODEL_TYPE,
    n_neighbors=N_NEIGHBORS,
)

print(f"Model trained: {MODEL_TYPE}")

In [ ]:
# Predict smoothed parameters
smoothed_params = predict_parameters(model, ta_data)

print(f"Smoothed parameters shape: {smoothed_params.shape}")

## Step 5: Compare Original vs ML-Smoothed Parameters

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# GSB amplitude
ax = axes[0]
ax.plot(time, fit_result.parameters[0, :], 'b.', alpha=0.5, label='Fitted')
ax.plot(time, smoothed_params[:, 0], 'r-', linewidth=2, label='ML smoothed')
ax.set_xlabel('Time (ps)')
ax.set_ylabel('Amplitude (mOD)')
ax.set_title('GSB Amplitude (461 nm)')
ax.legend()

# ESA1 amplitude
ax = axes[1]
ax.plot(time, fit_result.parameters[3, :], 'b.', alpha=0.5, label='Fitted')
ax.plot(time, smoothed_params[:, 3], 'r-', linewidth=2, label='ML smoothed')
ax.set_xlabel('Time (ps)')
ax.set_ylabel('Amplitude (mOD)')
ax.set_title('ESA1 Amplitude (600 nm)')
ax.legend()

# ESA2 amplitude
ax = axes[2]
ax.plot(time, fit_result.parameters[6, :], 'b.', alpha=0.5, label='Fitted')
ax.plot(time, smoothed_params[:, 6], 'r-', linewidth=2, label='ML smoothed')
ax.set_xlabel('Time (ps)')
ax.set_ylabel('Amplitude (mOD)')
ax.set_title('ESA2 Amplitude (680 nm)')
ax.legend()

plt.tight_layout()
plt.show()

## Step 6: Reconstruct Denoised Spectra

In [ ]:
# Reconstruct all spectra from smoothed parameters
reconstructed_signal = np.zeros_like(ta_data.signal)

for itime in range(len(time)):
    reconstructed_signal[:, itime] = reconstruct_spectrum(
        wavelength, smoothed_params[itime, :]
    )

reconstructed_data = TAData(
    wavelength=wavelength,
    time=time,
    signal=reconstructed_signal,
)

In [ ]:
# Compare original and reconstructed
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_ta_contour(ta_data, ax=axes[0])
axes[0].set_title('Original (noisy)')

plot_ta_contour(reconstructed_data, ax=axes[1])
axes[1].set_title('Reconstructed (ML smoothed)')

plt.tight_layout()
plt.show()

In [ ]:
# Compare spectra at specific times
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, itime in zip(axes, [5, 20, 50]):
    ax.plot(wavelength, ta_data.signal[:, itime], 'k-', alpha=0.5, label='Original')
    ax.plot(wavelength, reconstructed_signal[:, itime], 'r-', linewidth=2, label='Reconstructed')
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.5)
    ax.set_xlabel('Wavelength (nm)')
    ax.set_ylabel('ΔA (mOD)')
    ax.set_title(f't = {time[itime]:.1f} ps')
    ax.legend()

plt.tight_layout()
plt.show()

## Summary

### Workflow
1. **Fit Gaussians** - Extract parameters at each time point
2. **Train ML model** - Learn relationship between signal and parameters
3. **Predict smoothed parameters** - Apply model to get consistent values
4. **Reconstruct spectra** - Build denoised data from parameters

### Parameter Interpretation
- **Amplitude** → Population of species (kinetics)
- **Center** → Spectral position (should be ~constant)
- **Width** → Homogeneous/inhomogeneous broadening

### Model Choice
- **KNN**: Good for smooth interpolation, fast
- **Random Forest**: Better for noisy data, more robust

### Next Steps
- Fit kinetic models to amplitude vs time
- Extract rate constants
- Compare with Kinetiscope simulations